# Sentiment Analysis and Topic Modeling of US Presidential Debates from 1960 to 2024

### Daniel Henke (xxx), Sven Kurth (xxx), Margherita Grosso (xxx), Luca Gudi (xxx)

## Install and Load Packages

In [14]:
!pip install -q gensim 
!pip install -q nltk
!pip install -q spacy
!python -m spacy download en_core_web_sm --quiet
!pip install -q transformers
!pip install -q torch
!pip install -q hf_xet
!pip install -q numpy
%pip uninstall -y gensim numpy > /dev/null 2>&1
%pip install -q numpy==1.26.4
!pip install -q gensim

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Note: you may need to restart the kernel to use updated packages.


The system cannot find the path specified.


Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
from nltk.tokenize import TreebankWordTokenizer
from nltk.sentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import hf_xet
import spacy

In [17]:
# Completely remove incompatible packages
%pip uninstall -y gensim scipy numpy

# Install known-compatible versions
%pip install numpy==1.26.4 scipy==1.10.1 gensim==4.3.2

Found existing installation: gensim 4.3.2
Uninstalling gensim-4.3.2:
  Successfully uninstalled gensim-4.3.2
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following yanked versions: 1.11.0, 1.14.0rc1
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Python >=3.7,<3.10; 1.7.0 Requires-Python >=3.7,<3.10; 1.7.1 Requires-Python >=3.7,<3.10; 1.7.2 Requires-Python >=3.7,<3.11; 1.7.3 Requires-Python >=3.7,<3.11; 1.8.0 Requires-Python >=3.8,<3.11; 1.8.0rc1 Requires-Python >=3.8,<3.11; 1.8.0rc2 Requires-Python >=3.8,<3.11; 1.8.0rc3 Requires-Python >=3.8,<3.11; 1.8.0rc4 Requires-Python >=3.8,<3.11; 1.8.1 Requires-Python >=3.8,<3.11; 1.9.0 Requires-Python >=3.8,<3.12; 1.9.0rc1 Requires-Python >=3.8,<3.12; 1.9.0rc2 Requires-Pyth

In [19]:
!pip install gensim

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
   ---------------------------------------- 0.0/24.0 MB ? eta -:--:--
   ------- -------------------------------- 4.7/24.0 MB 25.9 MB/s eta 0:00:01
   --------------------- ------------------ 12.8/24.0 MB 33.6 MB/s eta 0:00:01
   ---------------------------------- ----- 21.0/24.0 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------- 24.0/24.0 MB 31.7 MB/s eta 0:00:00
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
   ---------------------------------------- 0.0/45.9 MB ? eta -:--:--
   ----- ---------------------------------- 6.8/45.9 MB 34.9 MB/s eta 0:00:02
   ------------- -------------------------- 15.5/45.9 MB 37.4 MB/s eta 0:00:01
   -------------------- ------------------- 23.9/45.9 MB 37.7 MB/s eta 0:00:01
   --------------------------- ------------ 31.7/45.9 MB 38.0 MB/s eta 0:00:01
   ----------------------------------- ---- 40.4/45.9 MB 38.9 MB/s eta 0:00:01
   ---------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires packaging<24,>=16.8, but you have packaging 24.2 which is incompatible.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 5.29.4 which is incompatible.


In [20]:
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel, LdaSeqModel
from collections import Counter

RuntimeError: Compiled extensions are unavailable. If you've installed from a package, ask the package maintainer to include compiled extensions. If you're building Gensim from source yourself, install Cython and a C compiler, and then run `python setup.py build_ext --inplace` to retry. 

# Load and Prepare Data

## Create Dataframe for US Debates from 1960 to 2020 the Dataset from Kaggle

### Define Directory

In [22]:
DATA_DIRECTORY = "data_dir"

### Meta Data

In [23]:
INCUMBENT_PAIRS = {
    ("ford", "1976"), ("carter", "1980"), ("reagan", "1984"), ("bush", "1984"),
    ("bush", "1992"), ("quayle", "1992"), ("clinton", "1996"), ("gore", "1996"),
    ("bush", "2004"), ("cheney", "2004"), ("obama", "2012"), ("biden", "2012"),
    ("trump", "2020"), ("pence", "2020")
}

WINNER_PAIRS = {
    ("kennedy", "1960"), ("carter", "1976"), ("reagan", "1980"), ("bush", "1980"),
    ("reagan", "1984"), ("bush", "1984"), ("bush", "1988"), ("quayle", "1988"),
    ("clinton", "1992"), ("gore", "1992"), ("clinton", "1996"), ("gore", "1996"),
    ("bush", "2000"), ("cheney", "2000"), ("bush", "2004"), ("cheney", "2004"),
    ("obama", "2008"), ("biden", "2008"), ("obama", "2012"), ("biden", "2012"),
    ("trump", "2016"), ("pence", "2016"), ("biden", "2020"), ("harris", "2020")
}

CANDIDATES = {
    # Presidential
    "kennedy": "Democratic", "nixon": "Republican", "ford": "Republican",
    "carter": "Democratic", "reagan": "Republican", "anderson": "Independent",
    "mondale": "Democratic", "bush": "Republican", "dukakis": "Democratic",
    "clinton": "Democratic", "perot": "Independent", "dole": "Republican",
    "gore": "Democratic", "kerry": "Democratic", "obama": "Democratic",
    "mccain": "Republican", "romney": "Republican", "trump": "Republican",
    "biden": "Democratic",

    # Vice-Presidential
    "ferraro": "Democratic", "quayle": "Republican", "bentsen": "Democratic",
    "kemp": "Republican", "stockdale": "Independent", "lieberman": "Democratic", "cheney": "Republican",
    "edwards": "Democratic", "palin": "Republican", "ryan": "Republican",
    "kaine": "Democratic", "pence": "Republican", "harris": "Democratic",
    "vance": "Republican"
}

VP_CORRECTIONS = ["2000-10-11", "2008-09-26", "2012-10-03"]

### Functions for Data Cleaning

In [24]:
def normalize_last_name(full_name):
    """Extracts the last name and applies capitalization."""
    if not full_name or full_name == "UNKNOWN":
        return "UNKNOWN"
    return full_name.strip().split()[-1].capitalize()

def fix_duplicate_names(last_name, year, is_candidate, party):
    """Fixes duplicate names for specific cases."""
    if last_name.lower() == "bush":
        last_name = "Bush Sr" if str(year) in ["1984", "1988", "1992"] else "Bush Jr"
    elif last_name.lower() == "clinton":
        last_name = "Clinton Hillary" if str(year) == "2016" else "Clinton Bill"
    elif last_name.lower() == "edwards" and str(year) == "1960":
        is_candidate = False
        party = None
    return last_name, is_candidate, party

### Functions for Data Enrichment

In [25]:
def check_incumbent(last_name, year):
    """Returns True if candidate is incumbent that year."""
    return (last_name.lower(), str(year)) in INCUMBENT_PAIRS

def check_winner(last_name, year):
    """Returns True if candidate won in that year."""
    return (last_name.lower(), str(year)) in WINNER_PAIRS

def check_candidate(last_name):
    """Returns whether person was a candidate and their party if applicable."""
    key = last_name.lower()
    return (key in CANDIDATES), CANDIDATES.get(key)

def is_vp_debate(content):
    """Determines if a debate is a Vice-Presidential debate."""
    dialogues = [entry.get("dialogue", "").lower() for entry in content[:5]]
    return any("vice presidential" in d for d in dialogues)

def correct_vp_debate_flags(df):
    """Manually corrects VP debate flags for known false positives."""
    for d in VP_CORRECTIONS:
        df.loc[df["date"] == pd.to_datetime(d).date(), "VP_debate"] = False
    return df

def generate_debate_titles(df):
    """Add a 'debate_title' column to the DataFrame."""
    debate_titles = {}
    debate_counter = defaultdict(Counter)

    for date, group in df.groupby("date"):
        year = group["year"].iloc[0]
        is_vp = group["VP_debate"].iloc[0]
        
        # Get sorted unique candidate last names
        candidate_names = sorted(set(group[group["is_candidate"]]["actor"]))
        title_base = f"{year}_" + "_".join(candidate_names)

        if is_vp:
            full_title = f"{title_base}_VP"
        else:
            # Number the debate among similar candidate sets in same year
            debate_counter[year][title_base] += 1
            count = debate_counter[year][title_base]
            full_title = f"{title_base}_{count}"

        debate_titles[date] = full_title

    df["debate_title"] = df["date"].map(debate_titles)
    return df

### Functions for Loading Data from Directory

In [26]:
def get_json_files(directory):
    """Retrieves JSON files excluding partials from a directory."""
    return [
        os.path.join(directory, f)
        for f in os.listdir(directory)
        if f.endswith(".json") and not f.startswith("part")
    ]
    
def parse_date(date_list):
    """Converts a date list into a datetime.date object or returns 'UNKNOWN'."""
    try:
        return pd.to_datetime(" ".join(date_list)).date()
    except Exception:
        return pd.NaT

### Final Pipeline Functions for Data Loading and Dataframe Creation

In [27]:
def process_debate_file(file_path):
    """Processes a single JSON debate file and returns a list of parsed rows."""
    rows = []
    with open(file_path, "r", encoding="utf-8") as f:
        debate = json.load(f)
        content = debate.get("content", [])
        date = parse_date(debate.get("date", []))
        year = date.year if pd.notnull(date) else "UNKNOWN"
        vp_flag = is_vp_debate(content)

        for entry in content:
            actor_raw = entry.get("actor", "UNKNOWN")
            dialogue = entry.get("dialogue", "")
            last_name = normalize_last_name(actor_raw)

            is_candidate, party = check_candidate(last_name)
            is_incumbent = check_incumbent(last_name, year)
            is_winner = check_winner(last_name, year)

            # Fix duplicate names and candidate status
            last_name, is_candidate, party = fix_duplicate_names(last_name, year, is_candidate, party)

            rows.append({
                "debate_title" : None,
                "date": date,
                "year": year,
                "actor": last_name,
                "dialogue": dialogue,
                "is_candidate": is_candidate,
                "party": party,
                "is_winner": is_winner,
                "VP_debate": vp_flag,
                "is_incumbent": is_incumbent
            })
    return rows
    
def debates_to_dataframe(directory):
    """Converts debate JSON files to a DataFrame."""
    
    all_rows = []
    json_files = get_json_files(directory)

    for file_path in json_files:
        all_rows.extend(process_debate_file(file_path))

    df = pd.DataFrame(all_rows)
    df = correct_vp_debate_flags(df)
    df = generate_debate_titles(df)
    return df

In [28]:
df_debates=debates_to_dataframe(DATA_DIRECTORY)
df_debates.head()

,debate_title,date,year,actor,dialogue,is_candidate,party,is_winner,VP_debate,is_incumbent
0,2020_Harris_Pence_VP,2020-10-07,2020,Participants,Senator Kamala Harris (D-CA) and,False,None,False,True,False
1,2020_Harris_Pence_VP,2020-10-07,2020,Moderator,Susan Page (USA Today),False,None,False,True,False
2,2020_Harris_Pence_VP,2020-10-07,2020,Page,Good evening. From the University of Utah in S...,False,None,False,True,False
3,2020_Harris_Pence_VP,2020-10-07,2020,Pence,Thank you.,True,Republican,False,True,True
4,2020_Harris_Pence_VP,2020-10-07,2020,Page,Senator Harris and Vice President Pence thank ...,False,None,False,True,False


## Add 1992 and 2024 Debates from txt.files to Dataframe

### Function for Loading txt.files from Directory with Data Enrichment

In [29]:
def extract_debate_txt(file_path, title, year, date, vp_debate, candidate_info):
    """Extracts structured debate data from a transcript text file.
    Args:
        file_path (str): Path to the transcript text file.
        title (str): Title of the debate.
        year (int): Year of the debate.
        date (str): Date of the debate in 'YYYY-MM-DD' format.
        vp_debate (bool): Whether this is a vice-presidential debate.
        candidate_info (dict): Dictionary mapping speaker last names to:
            {"is_candidate": bool, "party": str, "is_winner": bool, "is_incumbent": bool}
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    date = pd.to_datetime(date, errors="coerce").date() if date else None
    pattern = re.compile(r'^([A-Z][A-Z\s.\-]*)(?:, [A-Z\s.]+)?:\s*(.*)')

    data, current_actor, current_text = [], None, []

    def append_block(actor, text):
        if not actor or not text:
            return
        info = candidate_info.get(actor, {
            'is_candidate': False, 'party': None,
            'is_winner': False, 'is_incumbent': False
        })
        data.append({
            "debate_title": title,  "date": date, "year": year,
            "actor": actor, "dialogue": ' '.join(text).strip(),
            "is_candidate": info['is_candidate'], "party": info['party'],
            "is_winner": info['is_winner'], "VP_debate": vp_debate,
            "is_incumbent": info['is_incumbent']
        })

    for line in lines:
        match = pattern.match(line)
        if match:
            append_block(current_actor, current_text)
            current_actor = match.group(1).split()[-1].title()
            current_text = [match.group(2)] if match.group(2) else []
        else:
            current_text.append(line)

    append_block(current_actor, current_text)
    return pd.DataFrame(data)

### Complete 1992 debate

In [30]:
debate_1992_first_half=extract_debate_txt(
    file_path="data_dir/transcript_1992_oct_15_first_half.txt",
    title="1992_Bush Sr_Clinton Bill_Perot_2",
    year=1992, date="1992-10-15", vp_debate=False,
    candidate_info={
        "Bush": {"is_candidate": True, "party": "Republican","is_winner": False,"is_incumbent": True},
        "Clinton": {"is_candidate": True,"party": "Democratic","is_winner": True,"is_incumbent": False},
        "Perot": {"is_candidate": True,"party": "Independent","is_winner": False,"is_incumbent": False}
    }
)

#Rename Bush to Bush Sr and Clinton to Clinton (Bill)
debate_1992_first_half.loc[debate_1992_first_half["actor"] == "Bush", "actor"] = "Bush Sr"
debate_1992_first_half.loc[debate_1992_first_half["actor"] == "Clinton", "actor"] = "Clinton Bill"
debate_1992_first_half.head(20)

,debate_title,date,year,actor,dialogue,is_candidate,party,is_winner,VP_debate,is_incumbent
0,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,Good evening and welcome to this second of thr...,False,None,False,False,False
1,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Bush Sr,Let’s go.,True,Republican,False,False,True
2,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,And I think the first question is over here.,False,None,False,False,False
3,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Question,Yes. I’d like to direct my question to Mr. Per...,False,None,False,False,False
4,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Perot,That’s right at the top of my agenda. We’ve sh...,True,Independent,False,False,False
5,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,"Thank you, Mr. Perot. I see that the president...",False,None,False,False,False
6,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Bush Sr,"Carole, the thing that saved us in this global...",True,Republican,False,False,True
7,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,Governor Clinton.,False,None,False,False,False
8,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Clinton Bill,"I’d like to answer the question, because I’ve ...",True,Democratic,True,False,False
9,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,Thank you. I think we have a question over here.,False,None,False,False,False


In [32]:
#concat with the rest of the data
df_debates = pd.concat([ debate_1992_first_half, df_debates], ignore_index=True)
df_debates.head(20)

,debate_title,date,year,actor,dialogue,is_candidate,party,is_winner,VP_debate,is_incumbent
0,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,Good evening and welcome to this second of thr...,False,None,False,False,False
1,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Bush Sr,Let’s go.,True,Republican,False,False,True
2,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,And I think the first question is over here.,False,None,False,False,False
3,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Question,Yes. I’d like to direct my question to Mr. Per...,False,None,False,False,False
4,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Perot,That’s right at the top of my agenda. We’ve sh...,True,Independent,False,False,False
5,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,"Thank you, Mr. Perot. I see that the president...",False,None,False,False,False
6,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Bush Sr,"Carole, the thing that saved us in this global...",True,Republican,False,False,True
7,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,Governor Clinton.,False,None,False,False,False
8,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Clinton Bill,"I’d like to answer the question, because I’ve ...",True,Democratic,True,False,False
9,1992_Bush Sr_Clinton Bill_Perot_2,1992-10-15,1992,Simpson,Thank you. I think we have a question over here.,False,None,False,False,False


### Add 2024 Debates

In [33]:
debate_2024_biden=extract_debate_txt(
    file_path="data_dir/transcript_2024_Trump_Biden.txt",
    title="2024_Trump_Biden",
    year=2024,date="2024-07-27",vp_debate=False,
    candidate_info={
        "Trump": {"is_candidate": True,"party": "Republican","is_winner": True,"is_incumbent": False},
        "Biden": {"is_candidate": True,"party": "Democratic","is_winner": False,"is_incumbent": True}
    }
)

df_debates = pd.concat([df_debates, debate_2024_biden], ignore_index=True)
debate_2024_biden.head(20)

,debate_title,date,year,actor,dialogue,is_candidate,party,is_winner,VP_debate,is_incumbent
0,2024_Trump_Biden,2024-07-27,2024,Tapper,"We’re live from Georgia, a key battleground st...",False,None,False,False,False
1,2024_Trump_Biden,2024-07-27,2024,Bash,This debate is being produced by CNN and it’s ...,False,None,False,False,False
2,2024_Trump_Biden,2024-07-27,2024,Tapper,"I’m Jake Tapper, anchor of CNN’s “The Lead” an...",False,None,False,False,False
3,2024_Trump_Biden,2024-07-27,2024,Bash,"When it’s time for a candidate to speak, his m...",False,None,False,False,False
4,2024_Trump_Biden,2024-07-27,2024,Tapper,Now please welcome the 46th president of the U...,False,None,False,False,False
5,2024_Trump_Biden,2024-07-27,2024,Biden,How are you? Good to be here. Thank you.,True,Democratic,False,False,True
6,2024_Trump_Biden,2024-07-27,2024,Tapper,And please welcome the 45th president of the U...,False,None,False,False,False
7,2024_Trump_Biden,2024-07-27,2024,Biden,You have to take a look at what I was left whe...,True,Democratic,False,False,True
8,2024_Trump_Biden,2024-07-27,2024,Tapper,Thank you. President Trump?,False,None,False,False,False
9,2024_Trump_Biden,2024-07-27,2024,Trump,We had the greatest economy in the history of ...,True,Republican,True,False,False


In [34]:
debate_2024_harris=extract_debate_txt(
    file_path="data_dir/transcript_2024_Trump_Harris.txt",
    title="2024_Trump_Harris",
    year=2024,date="2024-09-10",vp_debate=False,
    candidate_info={
        "Trump": {"is_candidate": True,"party": "Republican","is_winner": True,"is_incumbent": False},
        "Harris": {"is_candidate": True,"party": "Democratic","is_winner": False,"is_incumbent": False}
    }
)

df_debates = pd.concat([df_debates, debate_2024_harris], ignore_index=True)
debate_2024_harris.head(20)

,debate_title,date,year,actor,dialogue,is_candidate,party,is_winner,VP_debate,is_incumbent
0,2024_Trump_Harris,2024-09-10,2024,Muir,"Tonight, the high-stakes showdown here in Phil...",False,None,False,False,False
1,2024_Trump_Harris,2024-09-10,2024,Davis,A historic race for president upended just wee...,False,None,False,False,False
2,2024_Trump_Harris,2024-09-10,2024,Muir,The candidates separated by the smallest of ma...,False,None,False,False,False
3,2024_Trump_Harris,2024-09-10,2024,Muir,"Good evening, I'm David Muir. And thank you fo...",False,None,False,False,False
4,2024_Trump_Harris,2024-09-10,2024,Davis,And I'm Linsey Davis. Tonight's meeting could ...,False,None,False,False,False
5,2024_Trump_Harris,2024-09-10,2024,Muir,And that brings us to the rules of tonight's d...,False,None,False,False,False
6,2024_Trump_Harris,2024-09-10,2024,Davis,President Trump won the coin toss. He chose to...,False,None,False,False,False
7,2024_Trump_Harris,2024-09-10,2024,Muir,So let's now welcome the candidates to the sta...,False,None,False,False,False
8,2024_Trump_Harris,2024-09-10,2024,Harris,Kamala Harris. Let's have a good debate.,True,Democratic,False,False,False
9,2024_Trump_Harris,2024-09-10,2024,Trump,Nice to see you. Have fun.,True,Republican,True,False,False


In [35]:
debate_2024_vp=extract_debate_txt(
    file_path="data_dir/transcript_2024_Vance_Walz.txt",
    title="2024_Vance_Walz_VP",
    year=2024,date="2024-10-01",vp_debate=True,
    candidate_info={
        "Jdv": {"is_candidate": True,"party": "Republican","is_winner": True,"is_incumbent": False},
        "Tw": {"is_candidate": True,"party": "Democratic","is_winner": False,"is_incumbent": False}
    }
)

#Rename Jdv to Vance, Tw to Walz, No to O'Donnell, and Mb to Brennan
debate_2024_vp.loc[debate_2024_vp["actor"] == "Jdv", "actor"] = "Vance"
debate_2024_vp.loc[debate_2024_vp["actor"] == "Tw", "actor"] = "Walz"
debate_2024_vp.loc[debate_2024_vp["actor"] == "No", "actor"] = "O'Donnell"
debate_2024_vp.loc[debate_2024_vp["actor"] == "Mb", "actor"] = "Brennan"


df_debates = pd.concat([df_debates, debate_2024_vp], ignore_index=True)
debate_2024_vp.head(20)

,debate_title,date,year,actor,dialogue,is_candidate,party,is_winner,VP_debate,is_incumbent
0,2024_Vance_Walz_VP,2024-10-01,2024,O'Donnell,Good evening. I'm Norah O'Donnell and thank yo...,False,None,False,True,False
1,2024_Vance_Walz_VP,2024-10-01,2024,Brennan,I'm Margaret Brennan. In order to have a thoug...,False,None,False,True,False
2,2024_Vance_Walz_VP,2024-10-01,2024,Brennan,"Thank you, Norah. Earlier today, Iran launched...",False,None,False,True,False
3,2024_Vance_Walz_VP,2024-10-01,2024,Walz,"Well, thank you. And thank you for those joini...",True,Democratic,False,True,False
4,2024_Vance_Walz_VP,2024-10-01,2024,Brennan,"Governor, your time is up. Senator Vance, the ...",False,None,False,True,False
5,2024_Vance_Walz_VP,2024-10-01,2024,Vance,"So, Margaret, I want to answer the question. F...",True,Republican,True,True,False
6,2024_Vance_Walz_VP,2024-10-01,2024,Brennan,"Thank you, Senator. Governor Walz, do you care...",False,None,False,True,False
7,2024_Vance_Walz_VP,2024-10-01,2024,Walz,"Well, look, Donald Trump was in office. We'll ...",True,Democratic,False,True,False
8,2024_Vance_Walz_VP,2024-10-01,2024,Brennan,"Senator Vance, the U.S. did have a diplomatic ...",False,None,False,True,False
9,2024_Vance_Walz_VP,2024-10-01,2024,Vance,"Well, first of all, Margaret, diplomacy is not...",True,Republican,True,True,False


In [37]:
df_debates.sort_values(by=["year", "date"], inplace=True)
df_debates.reset_index(drop=True, inplace=True)
df_debates

,debate_title,date,year,actor,dialogue,is_candidate,party,is_winner,VP_debate,is_incumbent
0,1960_Kennedy_Nixon_1,1960-09-26,1960,Kennedy,"Mr. Smith, Mr. Nixon. In the election of 1860,...",True,Democratic,True,False,False
1,1960_Kennedy_Nixon_1,1960-09-26,1960,Smith,And now the opening statement by Vice Presiden...,False,None,False,False,False
2,1960_Kennedy_Nixon_1,1960-09-26,1960,Nixon,"Mr. Smith, Senator Kennedy. The things that Se...",True,Republican,False,False,False
3,1960_Kennedy_Nixon_1,1960-09-26,1960,Smith,"Thank you, Mr. Nixon. That completes the openi...",False,None,False,False,False
4,1960_Kennedy_Nixon_1,1960-09-26,1960,Fleming,"Senator, the Vice President in his campaign ha...",False,None,False,False,False
...,...,...,...,...,...,...,...,...,...,...
10212,2024_Vance_Walz_VP,2024-10-01,2024,Walz,"Well, thank you, Senator Vance. Thank you to C...",True,Democratic,False,True,False
10213,2024_Vance_Walz_VP,2024-10-01,2024,Brennan,"Governor Walz. Thank you. Senator Vance, your ...",False,None,False,True,False
10214,2024_Vance_Walz_VP,2024-10-01,2024,Vance,"Well, I want to thank Governor Walz, you folks...",True,Republican,True,True,False
10215,2024_Vance_Walz_VP,2024-10-01,2024,Brennan,"Senator Vance, thank you. And thank you both f...",False,None,False,True,False


### Save as CSV

In [ ]:
# Save the final DataFrame to a CSV file
df_debates.to_csv("debate_transcripts_cleaned.csv", index=False, encoding="utf-8")
print(f"Data saved")

# Exploratory Data Analysis

In [39]:
df_debate = pd.read_csv('debate_transcripts_cleaned.csv')

In [40]:
df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
sns.histplot(df_debate['word_count'], bins=50, color='#4C72B0', kde=True)
plt.title('Distribution of Sentence Lengths (in Words)')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

NameError: name 'sns' is not defined

<Figure size 1000x600 with 0 Axes>

In [ ]:
# Keep only candidates and filter outliers above 600 words
df_candidates = df_debate[df_debate['is_candidate'] == True].copy()
df_candidates['word_count'] = df_candidates['dialogue'].apply(lambda x: len(str(x).split()))

# Create 3-year election blocks
def map_year_block(y):
    if y < 2000:
        return "1960–1996"
    elif y <= 2012:
        return "2000–2012"
    else:
        return "2016–2024"

df_candidates['year_block'] = df_candidates['year'].apply(map_year_block)
# Plot
g = sns.displot(
    data=df_candidates,
    x='word_count',
    col='year_block',
    col_order=["1960–1996", "2000–2012", "2016–2024"],
    col_wrap=3,
    height=4,
    aspect=1.4,
    bins=40,
    kde=True,
    color='#4C72B0',
    facet_kws={'sharey': True}
)

g.set_titles("Period: {col_name}")
g.set_axis_labels("Words per Sentence", "Frequency")
plt.subplots_adjust(top=0.85)
g.fig.suptitle("Sentence Length Distribution", fontsize=16)
plt.show()

In [ ]:
# Make sure word_count exists
df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))

# Overall average sentence length
average_sentence_length = df_debate['word_count'].mean()
print(f"Average sentence length: {average_sentence_length:.2f} words")

In [ ]:
# Ensure word count column exists
df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))

# Filter only candidate speech
df_candidates = df_debate[df_debate['is_candidate'] == True]

# Calculate average sentence length
average_sentence_length = df_candidates['word_count'].mean()
print(f"Average sentence length (candidates only): {average_sentence_length:.2f} words")


In [ ]:
# Define your custom eras
def assign_time_span(year):
    if year <= 1980:
        return '1960–1980'
    elif year <= 2004:
        return '1984–2004'
    else:
        return '2008–2024'

df_candidates['time_span'] = df_candidates['year'].apply(assign_time_span)

# Compute average per span
avg_per_span = df_candidates.groupby('time_span')['word_count'].mean().reset_index(name='avg_sentence_length')
print(avg_per_span)

In [ ]:
words_per_candidate = df_debate[(df_debate['is_candidate'] == True) & (df_debate['VP_debate'] == False)].groupby('actor')['word_count'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
words_per_candidate.plot(kind='bar', color='#4C72B0')
plt.title('Total Words Spoken by Each Candidate')
plt.ylabel('Total Word Count')
plt.xlabel('Candidate')
plt.xticks(rotation=45, size=10)
plt.yticks(size=8.5)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import Patch

# Step 1: Filter presidential candidate data
df_debate['word_count'] = df_debate['dialogue'].apply(lambda x: len(str(x).split()))
pres_debate = df_debate[(df_debate['is_candidate'] == True) & (df_debate['VP_debate'] == False)]

# Step 2: Words per candidate per debate
words_per_debate = pres_debate.groupby(['actor', 'date'])['word_count'].sum().reset_index()
avg_words_per_candidate = words_per_debate.groupby('actor')['word_count'].mean().sort_values(ascending=False)

# Step 3: Map each candidate to their party (take the most frequent party per actor)
actor_party_map = pres_debate.groupby('actor')['party'].agg(lambda x: x.mode().iloc[0])

# Step 4: Get list of colors by actor
bar_colors = [party_palette.get(actor_party_map.get(actor, 'Independent'), '#808080') for actor in avg_words_per_candidate.index]

# Step 5: Plot
plt.figure(figsize=(8, 5))
avg_words_per_candidate.plot(
    kind='bar',
    color=bar_colors
)

legend_elements = [
    Patch(facecolor='#007FFF', label='Democratic'),
    Patch(facecolor='#d62728', label='Republican'),
    Patch(facecolor='#FFBF00', label='Independent')
]
plt.legend(handles=legend_elements, title='Party', loc='upper right')

plt.title('Average Words per Presidential Debate by Candidate', size=14)
plt.ylabel('Avg. Word Count per Debate')
plt.xlabel('Candidate')
plt.xticks(rotation=60, size=9)
plt.yticks(size=9)
plt.tight_layout()
plt.show()

In [ ]:
# Filter candidates only
df_candidates = df_debate[df_debate['is_candidate'] == True].copy()
df_candidates['word_count'] = df_candidates['dialogue'].apply(lambda x: len(str(x).split()))
#df_candidates = df_candidates[df_candidates['word_count'] <= 600] 

# Custom party color palette 
party_palette = {
    'Democratic': '#007FFF',   
    'Republican': '#d62728',    
    'Independent': '#FFBF00'   
}

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df_candidates,
    x='party',
    y='word_count',
    palette=party_palette,
    showfliers=True,
    width=0.6
)

plt.title('Sentence Length by Party', fontsize=16)
plt.xlabel('Party', fontsize=12)
plt.ylabel('Word Count', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

# Sentiment Analysis

# Topic Modeling

## Dynamic Topic Modeling using LDASeqModel() from Gensim

## Topic Modeling using KeyBERT and BERTopic